# Attention Head Cooperation

Analyze how heads work together: within-layer cooperation, cross-layer pipelines,
output diversity, contribution ranking, and cooperation summary.

## Why This Matters

Attention head cooperation reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_cooperation import (
    within_layer_cooperation, cross_layer_head_pipeline,
    head_output_diversity, head_contribution_ranking,
    cooperation_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Within-Layer Cooperation

Are heads in the same layer redundant, complementary, or competing?

In [ ]:
for layer in range(4):
    result = within_layer_cooperation(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_redundant']} redundant, {result['n_complementary']} complementary")
    for p in result['pairs'][:3]:
        print(f"    H{p['head_i']}<->H{p['head_j']}: out_cos={p['output_cosine']:.3f}, "
              f"attn_cos={p['attention_cosine']:.3f} [{p['relation']}]")

## Cross-Layer Pipeline

Do layer outputs feed into next layer's attention?

In [ ]:
result = cross_layer_head_pipeline(model, tokens)
for t in result['per_transition']:
    print(f"  L{t['from_layer']} -> L{t['to_layer']}: "
          f"out_norm={t['output_norm']:.4f}, next_entropy={t['next_layer_entropy']:.4f}")

## Head Output Diversity

Are head outputs diverse or similar within each layer?

In [ ]:
for layer in range(4):
    result = head_output_diversity(model, tokens, layer=layer)
    flag = ' [DIVERSE]' if result['is_diverse'] else ''
    print(f"  Layer {layer}: cos={result['mean_pairwise_cosine']:.4f}, "
          f"diversity={result['mean_diversity']:.4f}{flag}")

## Head Contribution Ranking

Which heads contribute most to the final prediction?

In [ ]:
result = head_contribution_ranking(model, tokens)
print(f"Most impactful: {result['most_impactful']}\n")
for h in result['heads'][:8]:
    bar = '█' * int(min(h['logit_range'] * 5, 30))
    print(f"  L{h['layer']}H{h['head']}: range={h['logit_range']:.4f}, "
          f"top_tok={h['top_promoted_token']} {bar}")

## Cooperation Summary

Cross-layer overview of head cooperation patterns.

In [ ]:
result = cooperation_summary(model, tokens)
for p in result['per_layer']:
    flag = ' [DIVERSE]' if p['is_diverse'] else ''
    print(f"  Layer {p['layer']}: diversity={p['mean_diversity']:.4f}, "
          f"redundant={p['n_redundant_pairs']}, "
          f"complementary={p['n_complementary_pairs']}{flag}")